# 01 · Exploratory data analysis
Reads from the `analytics` views, the same source Power BI and Excel use. Run from the repo root
(`jupyter lab` in the repo, or set the working directory) with `DATABASE_URL` in `.env`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import matplotlib.pyplot as plt
from src.common import get_engine
engine = get_engine()
aff = pd.read_sql('SELECT * FROM analytics.v_affordability', engine, parse_dates=['quarter_end'])
aff.shape

## Melbourne-wide median rent by property type

In [ ]:
trend = (aff.groupby(['quarter_end', 'property_type'])['median_weekly_rent'].median().unstack())
trend.plot(figsize=(10, 5), title='Median of suburb median rents ($/week)');

## Latest quarter: rent-to-income vs distance to station

In [ ]:
latest = aff[aff.quarter_end == aff.quarter_end.max()].dropna(subset=['rent_to_income_pct'])
ax = latest.plot.scatter(x='distance_to_station_km', y='rent_to_income_pct', c='bedrooms', cmap='viridis',
                         figsize=(9, 5), alpha=0.7)
ax.axhline(30, ls='--', c='grey'); ax.set_title('Rent-to-income % (30% = rental stress line)');

## Cheapest rent-to-income within 1.5 km of a station (2 bed flats, latest quarter)

In [ ]:
(latest[(latest.distance_to_station_km <= 1.5) & (latest.property_type == '2 bed flat') & latest.is_reliable]
   .sort_values('rent_to_income_pct')
   [['suburb_group_name', 'region', 'nearest_station', 'distance_to_station_km', 'median_weekly_rent',
     'median_weekly_hh_income', 'rent_to_income_pct', 'yoy_growth_pct']]
   .head(15))

## Where is growth accelerating? (YoY growth, latest vs a year earlier)

In [ ]:
yoy = aff.pivot_table(index='suburb_group_name', columns='quarter_end', values='yoy_growth_pct', aggfunc='median')
q = sorted(yoy.columns)
accel = (yoy[q[-1]] - yoy[q[-5]]).sort_values(ascending=False)
accel.head(10).to_frame('change in YoY growth (pp)')

## Share of suburb/type rows in rental stress over time

In [ ]:
stress = (aff.dropna(subset=['affordability_band'])
             .assign(stress=lambda d: d.affordability_band.str.startswith('Rental stress'))
             .groupby('quarter_end').stress.mean() * 100)
stress.plot(figsize=(10, 4), title='% of suburb x property type rows with rent >= 30% of indexed income');